# 07. MAF で構築したエージェントを Hosted Agent としてデプロイする

## コードの解説

**Hosted Agent** とは、MAF で構築したエージェントをコンテナ化して Foundry Agent Service 上で動作させる仕組みです。
エージェントの Python コードを `ResponsesHostServer` でラップして HTTP サーバーとして起動し、
その Dockerfile をビルドして Azure Container Registry (ACR) にプッシュ、最後に Foundry Agent Service へデプロイします。

### Hosted Agent のデプロイフロー

```
┌─────────────────────────────────────────────────────────────────┐
│  1. agent_app.py を作成                                          │
│     └─ MAF Agent を ResponsesHostServer でラップ → HTTP サーバー │
│                                                                  │
│  2. Dockerfile でコンテナ化                                       │
│     └─ python agent_app.py → ポート 8088 でリッスン              │
│                                                                  │
│  3. ACR へイメージをプッシュ                                      │
│     └─ az acr build ...                                          │
│                                                                  │
│  4. Foundry Agent Service へデプロイ                             │
│     └─ (definition: HostedAgentDefinition, image: <acr>/...) で登録│
└─────────────────────────────────────────────────────────────────┘
```

### ローカルエージェントとの違い

| 項目 | ローカル MAF エージェント | Hosted Agent |
|-----|----------------------|-------------|
| 実行場所 | ローカル Python プロセス | Azure クラウド（コンテナ） |
| 起動方法 | `await agent.run(...)` | `uvicorn` で HTTP サーバーを起動 |
| ホスティング層 | なし | `ResponsesHostServer` アダプター |
| デプロイ成果物 | なし | Docker イメージ → ACR |
| Foundry 登録 | なし | `kind: hosted` エージェント定義 |

### 使用するパッケージ

| パッケージ | 役割 |
|-----------|------|
| `agent-framework` | MAF コア（`Agent` クラスなど） |
| `agent-framework-foundry-hosting` | `ResponsesHostServer`（HTTP アダプター） |
| `uvicorn` | ASGI サーバー（コンテナ内で起動） |

## デプロイ手順の概要

### 1. agent_app.py の作成

エージェントを `ResponsesHostServer` でラップし、`uvicorn` で起動するエントリーポイントを作成します。

```python
import os
from starlette.requests import Request
from starlette.responses import Response
from agent_framework import Agent
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework_foundry_hosting import ResponsesHostServer


AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")
if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
else:
    from azure.identity import ManagedIdentityCredential
    auth_kwargs = dict(credential=ManagedIdentityCredential())


# ---------------------------------------------------------------
# MAF エージェントの定義
#
# ここは通常の MAF エージェントと同じ。
# Hosted Agent 化するにはこのエージェントを HTTP サーバーでラップするだけ。
# ---------------------------------------------------------------
agent = Agent(
    client=OpenAIChatCompletionClient(
        **auth_kwargs,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        model=MODEL_DEPLOYMENT_NAME,
    ),
    name="HostedTravelAssistant",
    instructions=(
        "あなたは Microsoft Foundry でホストされた旅行アシスタントです。"
        "旅行先のおすすめ情報や旅のコツをわかりやすく提供してください。"
    ),
)


# ---------------------------------------------------------------
# ResponsesHostServer でエージェントを HTTP サーバーとしてラップ
#
# ポイント1: ResponsesHostServer は ASGI アプリ（Starlette ベース）
# ポイント2: Foundry Agent Service の "responses" プロトコルに準拠した
#            エンドポイントを自動生成する
# ポイント3: コンテナ内では uvicorn でこの app を起動する
#            例: uvicorn agent_app:app --host 0.0.0.0 --port 8088
# ---------------------------------------------------------------
app = ResponsesHostServer(agent)
```

### 2. Dockerfile の作成

```dockerfile
FROM python:3.12-slim
WORKDIR /app
RUN pip install --no-cache-dir agent-framework agent-framework-openai agent-framework-foundry-hosting
COPY agent_app.py .
EXPOSE 8088
CMD ["uvicorn", "agent_app:app", "--host", "0.0.0.0", "--port", "8088"]
```

### 3. ACR の作成

Hosted Agent 用のコンテナイメージを保存する Azure Container Registry を作成します。

```bash
az acr create \
  --resource-group <RESOURCE_GROUP> \
  --name <ACR_NAME> \
  --sku Basic
```

### 4. Dockerfile によるリモートビルド

`az acr build` を使って ACR 上でイメージをビルドします。ローカルに Docker が不要なため、Dev Container 環境からそのまま実行できます。

```bash
az acr build \
  --registry <ACR_NAME> \
  --image hostedtravelassistant:v1 \
  .
```

### 5. Azure ロールの付与

Hosted Agent が所有する ID が Foundry プロジェクトにアクセスできるようにするため、エージェントに対してリソースグループのスコープで Azrue AI Developer ロールを付与します。

### 6. Python SDK による Hosted Agent の作成

`azure-ai-projects` SDK の `client.agents.create_version()` を使って Hosted Agent を登録します。

`container_protocol_versions` に `responses` プロトコルのバージョン `1.0.0` を明示指定することで、Foundry プラットフォームがコンテナの responses プロトコル対応を認識できます。


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)

load_dotenv()

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
ACR_NAME = os.getenv("ACR_NAME")

client = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

agent = client.agents.create_version(
    agent_name="HostedTravelAssistant",
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(
                protocol=AgentProtocol.RESPONSES,
                version="1.0.0",
            )
        ],
        cpu="1",
        memory="2Gi",
        image=f"{ACR_NAME}.azurecr.io/hostedtravelassistant:v1",
        environment_variables={
            "USE_KEY_AUTH": "True",
            "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
            "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
            "AZURE_OPENAI_CHAT_DEPLOYMENT": AZURE_OPENAI_CHAT_DEPLOYMENT,
        },
    ),
)
print(f"エージェント作成完了: {agent}")
